# CommGuard calibration v2

This notebook measures idle PCIe traffic and repeated collective payload groups at two sampling frequencies. It tests capture repeatability before interpreting aggregate correlation. NVML PCIe readings are not direct NCCL byte counts. A result may be `supported`, `partially_supported`, or `not_supported`; none of these results establishes training detection.


## Install a reviewed source revision

Set `GIT_REF` to a reviewed commit SHA for a research run. The install uses `--no-deps` and does not alter Kaggle's preinstalled PyTorch/CUDA stack. The notebook never deletes an existing working directory.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import importlib
import json
import site
import subprocess
import sys

REPO_URL = "https://github.com/waqasm86/CommGuard.git"
GIT_REF = "main"  # Replace with a reviewed commit SHA for research results.
REPO = Path("/kaggle/working/CommGuard")
ARTIFACTS = Path("/kaggle/working/commguard-artifacts")

if REPO.exists() and not (REPO / ".git").is_dir():
    raise RuntimeError(f"Refusing to replace non-Git directory: {REPO}")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", GIT_REF], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)
COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-build-isolation", "--no-deps", "-e", str(REPO)],
    check=True,
)
site.addsitedir(str(REPO / "src"))
importlib.invalidate_caches()
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print("Reviewed source commit:", COMMIT)


## Strict dual-T4 gate

This must report exactly two NVIDIA T4 GPUs, CUDA, NCCL, and distinct GPU identities before any experiment is enabled.


In [ ]:
from commguard.environment import check_environment

environment = check_environment(strict=True, output=ARTIFACTS)
assert environment["readiness"]["exactly_two_gpus"]
assert environment["readiness"]["both_t4"]
print("Strict dual-T4 readiness:", environment["strict_ready"])


## Run the repetition-aware calibration matrix

This is an expensive 60-run GPU experiment and is disabled by default. Review the configuration, set `RUN_CALIBRATION = True`, and rerun this cell. Each sampling frequency receives its own idle baseline and five repetitions per payload.


In [ ]:
from commguard.artifacts import ArtifactStore
from commguard.calibration import analyze_calibration
from commguard.orchestrator import run_experiment

def observation_from_outcome(outcome, payload_mib, sampling_hz, *, is_idle=False):
    return {
        "run_id": outcome["run_id"],
        "payload_mib": payload_mib,
        "sampling_hz": sampling_hz,
        "is_idle": is_idle,
        "participation_valid": outcome["manifest"]["participation_valid"],
        "pcie_supported": outcome["pcie_supported"],
        "pcie_total_mean_bytes_per_s": outcome["pcie_total_mean_bytes_per_s"],
    }

RUN_CALIBRATION = False
SAMPLING_HZ = (1, 5)
PAYLOAD_MIB = (1, 4, 16, 64, 256)
REPETITIONS = 5
PRIMARY_SAMPLING_HZ = 5

frequency_results = {}
if RUN_CALIBRATION:
    rows_by_hz = {hz: [] for hz in SAMPLING_HZ}
    for hz in SAMPLING_HZ:
        sampling_interval_s = 1 / hz
        for repetition in range(REPETITIONS):
            outcome = run_experiment(
                "control_idle",
                output=ARTIFACTS,
                overrides={
                    "repetition": repetition,
                    "iterations": 20,
                    "idle_interval_s": 0.25,
                    "sampling_interval_s": sampling_interval_s,
                },
                raise_on_failure=False,
            )
            rows_by_hz[hz].append(observation_from_outcome(outcome, 0, hz, is_idle=True))
        for payload in PAYLOAD_MIB:
            for repetition in range(REPETITIONS):
                outcome = run_experiment(
                    "collective_all_reduce_1mib",
                    output=ARTIFACTS,
                    overrides={
                        "payload_mib": payload,
                        "repetition": repetition,
                        "iterations": 20,
                        "burst_iterations": 100,
                        "iteration_interval_s": 0.25,
                        "sampling_interval_s": sampling_interval_s,
                    },
                    raise_on_failure=False,
                )
                rows_by_hz[hz].append(observation_from_outcome(outcome, payload, hz))
        frequency_results[str(hz)] = analyze_calibration(
            rows_by_hz[hz],
            minimum_repetitions=REPETITIONS,
            minimum_capture_rate=0.8,
        )

    store = ArtifactStore(ARTIFACTS)
    store.initialize()
    primary_result = frequency_results[str(PRIMARY_SAMPLING_HZ)]
    store.write_json("results/calibration-v2.json", primary_result)
    store.write_json("results/frequency-study.json", frequency_results, validate=False)
    print(json.dumps({hz: value["status"] for hz, value in frequency_results.items()}, indent=2))
else:
    print("Calibration skipped. Set RUN_CALIBRATION=True only after reviewing this cell.")


## Export the calibration evidence

After a completed run, this creates a new archive without overwriting an earlier export. Download it from Kaggle's Output panel. In the next session, upload it as a Kaggle Dataset and set `INPUT_BUNDLE` in `commguard_benign_corpus.ipynb` to its `/kaggle/input/...` path.


In [ ]:
if RUN_CALIBRATION:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    archive_path = Path("/kaggle/working") / f"commguard-calibration-{stamp}-{COMMIT[:12]}.tar.gz"
    exported = ArtifactStore(ARTIFACTS).export(archive_path)
    print("Download for the next notebook:", exported)
else:
    print("Nothing exported because calibration was not enabled.")
